In [30]:
import pandas as pd
import matplotlib.pyplot as plt 
import geopandas as gpd
import numpy as np
from bs4 import BeautifulSoup
import requests
import time

In [17]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [20]:
baseurl = 'https://www.vesselfinder.com/vessels'

shipcsv = '/Users/griffinberonio/Documents/Holloway_Group/Data/Copy of south_coast_ship_lookup.csv'

In [11]:
ships = pd.read_csv(shipcsv)
ships.head()


,imo,mmsi,vessel_name,vessel_group,vessel_type,VesselFinder,GT,Build_Year,Engine_Power_kW,Max_Speed_knots,Notes
0,9354662,256430000,A IDEFIX,Cargo,70,Open,NaN,NaN,NaN,NaN,NaN
1,7829364,367572390,A N TILLETT,Tug,31,Open,NaN,NaN,NaN,NaN,NaN
2,9188544,366755020,ADMIRAL,Tug,31,Open,NaN,NaN,NaN,NaN,NaN
3,7381398,441403000,AGNES 7,Fishing,30,Open,NaN,NaN,NaN,NaN,NaN
4,9891660,538009248,AIGEORGIS,Tanker,80,Open,NaN,NaN,NaN,NaN,NaN


In [46]:
from selenium.webdriver.common.action_chains import ActionChains

testship = '9354662'
# Spin up a Chrome browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

driver.get(baseurl)

# Find elements — same ideas as BeautifulSoup but different syntax

search_box = driver.find_element(By.ID, "advsearch-name")
print(search_box)
print(search_box.tag_name)       # should print "input"
print(search_box.is_displayed()) # should print True
print(search_box.is_enabled()) 

# driver.execute_script("arguments[0].value = arguments[1];", search_box, "9354662")
time.sleep(1)
ActionChains(driver).move_to_element(search_box).click().send_keys(testship).perform()
time.sleep(2)
time.sleep(1)
# search_box.send_keys('9354662')
search_button = driver.find_element(By.CSS_SELECTOR, "button[data-action='Search']")
search_button.click()

# Wait for results page to load
WebDriverWait(driver, 10).until(
    EC.url_changes(driver.current_url)
)

print(driver.title)
print(driver.page_source[:2000])

# Click the ship link
ship_link = driver.find_element(By.CSS_SELECTOR, "a.ship-link")
ship_link.click()

# Wait for the details page to load
WebDriverWait(driver, 10).until(
    EC.url_contains("/vessels/details/")
)

# Scrape the details page
soup = BeautifulSoup(driver.page_source, "html.parser")

<selenium.webdriver.remote.webelement.WebElement (session="1d3a369c7706f4e2a766df6dc4396304", element="f.B7E445AD1A7BEC7F349E9C780D9AF4F6.d.B1D66D8DDC934AA48EF354D71BE58323.e.2")>
input
True
True
Vessels Database - VesselFinder
<html lang="en"><head style="">
<script async="" type="text/javascript" src="https://btloader.com/tag?o=5708166709903360&amp;upapi=true"></script><script async="" src="//cdn.confiant-integrations.net/gptprebidnative/202606101730/wrap.js"></script><script async="" type="text/javascript" src="https://cdn.fuseplatform.net/prebid/IIQUniversalID-6.253.js"></script><script async="" type="text/javascript" src="https://cdn.confiant-integrations.net/JvyCyRZHfbxRlxVAXY4WNFGS07w/gpt_and_prebid/config.js"></script><script async="" type="text/javascript" src="https://7oDcS2a8NXtFyxnDn.ay.delivery/s2s-client-v1.js"></script><script async="" type="text/javascript" src="https://securepubads.g.doubleclick.net/tag/js/gpt.js"></script><script async="" src="//c.amazon-adsystem.com/

In [47]:
print(soup)

<html lang="en"><head style="">
<script async="" src="https://static.vesselfinder.net/web/tippy-all.4.min.js" type="text/javascript"></script><script async="" src="https://btloader.com/tag?o=5708166709903360&amp;upapi=true" type="text/javascript"></script><script async="" src="//cdn.confiant-integrations.net/gptprebidnative/202606101730/wrap.js"></script><script async="" src="https://cdn.fuseplatform.net/prebid/IIQUniversalID-6.253.js" type="text/javascript"></script><script async="" src="https://cdn.confiant-integrations.net/JvyCyRZHfbxRlxVAXY4WNFGS07w/gpt_and_prebid/config.js" type="text/javascript"></script><script async="" src="https://7oDcS2a8NXtFyxnDn.ay.delivery/s2s-client-v1.js" type="text/javascript"></script><script async="" src="https://securepubads.g.doubleclick.net/tag/js/gpt.js" type="text/javascript"></script><script async="" src="//c.amazon-adsystem.com/aax2/apstag.js"></script><script async="" src="https://cdn.fuseplatform.net/prebid/prebid-9.53.5-d5eff65d51a8d1b0a6b62

In [48]:
for section in soup.find_all("section", class_="ship-section"):
    heading = section.find("h2", class_="bar")
    if heading and heading.text.strip() == "Vessel Particulars":
        
        # Parse all label/value pairs into a dict
        data = {}
        for row in section.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) == 2:
                label = cells[0].text.strip()
                value = cells[1].text.strip()
                data[label] = value
        
        # Pull what you need
        gross_tonnage = data.get("Gross Tonnage")
        year_of_build = data.get("Year of Build")
        
        print(f"Gross Tonnage: {gross_tonnage}")
        print(f"Year of Build: {year_of_build}")
        break

Gross Tonnage: 18263
Year of Build: 2008


In [41]:

driver.quit() 

In [ ]:

headers = {
    
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
response = requests.get(baseurl, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")
soup

Error 404 (Not Found)

Unable to find matching target resource method